# NeMo GPU worker (Google Colab)

Runtime → Change runtime type → **T4 GPU**.

This notebook only runs STT/TTS. The dashboard, VAD, AudioSocket and LLM stay on your PC.

1. Run the install + server cells.
2. Copy the public URL into local `.env` as `REMOTE_INFERENCE_URL`.
3. On the PC: `STT_BACKEND=remote` and `TTS_BACKEND=remote`.

In [ ]:
!nvidia-smi -L
%pip install -q "nemo_toolkit[asr,tts]" fastapi uvicorn pyngrok soundfile python-multipart

In [ ]:
import os, pathlib, urllib.request
os.environ["INFERENCE_TOKEN"] = os.environ.get("INFERENCE_TOKEN", "lab-token-change-me")
raw = "https://raw.githubusercontent.com/haroldcastro-devai/realtime-voice-agent/main/colab/nemo_server.py"
pathlib.Path("nemo_server.py").write_bytes(urllib.request.urlopen(raw).read())
print("token set, server file downloaded")

In [ ]:
import threading, uvicorn
from pyngrok import ngrok
from nemo_server import app

# Optional: ngrok.set_auth_token("your-ngrok-token")
def _run():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")
threading.Thread(target=_run, daemon=True).start()
tunnel = ngrok.connect(8000, bind_tls=True)
print("REMOTE_INFERENCE_URL=" + tunnel.public_url)
print("REMOTE_INFERENCE_TOKEN=" + os.environ["INFERENCE_TOKEN"])